# Data Harmonisation

This notebook is the workflow-facing Step 2 harmonisation pipeline for the MOTEL platform. The implementation lives in `harmonise_helpers.py`; this notebook focuses on the process, the data handoff, and the outputs created during harmonisation.

> Renku note: this notebook is not intended to run in the standard Renku demo setup because the harmonisation step depends on a locally hosted LLM runtime via `ollama` and the local model `qwen3:14b`. In principle, a Renku deployment could run this workflow if it provides a compatible custom session image, enough CPU/GPU resources, and access to the local model service, but that is outside the default demo environment.

## Workflow

1. define run controls and repository paths
2. load schema context, staged unmapped data, and current registry state
3. prepare LLM guidance and optional setup controls
4. resolve entities and controlled vocabularies
5. build linked entities, write mapping files, and review the audit output


## Run Controls

Use these controls to choose whether the notebook resets staging status, backs up the database outputs, clears derived files, and limits the run to a small test slice.


In [ ]:
set_all_unmapped_to_pending = True
create_motel_db_backup = True
reset_motel_db_outputs = True
test_limit = None   # None means no limit, otherwise set to an integer
audit_indices = [0, 1, 2]

In [ ]:
import importlib

import harmonise_helpers as hh
importlib.reload(hh)

PATHS = hh.get_harmonisation_paths()

print(f"Project root: {PATHS['project_root']}")
print(f"Schema directory: {PATHS['schema_dir']}")
print(f"Staged unmapped input: {PATHS['unmapped_path']}")
print(f"Linked entity output: {PATHS['linked_entity_path']}")
print(f"Mapping directory: {PATHS['mapping_dir']}")

## Load Context

Harmonisation takes staged `unmapped_entity` records from Step 1 and resolves them into MOTEL registries, controlled vocabularies, mapping tables, and linked entities.


In [ ]:
all_schemas = hh.load_all_schemas(PATHS["schema_dir"])
inputs = hh.prepare_harmonisation_inputs(
    PATHS["unmapped_path"],
    test_limit=test_limit,
    set_all_unmapped_to_pending=set_all_unmapped_to_pending,
)

all_unmapped_entities = inputs["all_unmapped_entities"]
ue = inputs["ue"]
ue_indices = inputs["ue_indices"]

print(f"Loaded schemas: {len(all_schemas)}")
print(f"Staged entities: {len(all_unmapped_entities)}")
print(f"Selected for this run: {len(ue)}")
print(f"Already mapped: {len(all_unmapped_entities) - len(ue)}")

## Prepare Registry Context

This step uses the staged `unmapped_entity` outputs from Step 1 together with the current MOTEL registries. Workbook-derived context is expected to have been packaged during ingestion rather than loaded again here.


In [ ]:
import pandas as pd

hh.pd = pd  # inject pd into harmonise_helpers module globals

all_csv_data = hh.load_all_csv_data(PATHS["database_dir"])
harmonisation_started, harmonisation_log = hh.start_harmonisation_run(
    PATHS,
    all_schemas,
    all_unmapped_entities,
    ue,
    test_limit=test_limit,
)

print(f"Loaded CSV datasets: {len(all_csv_data)}")
print(f"Harmonisation log: {harmonisation_log}")

## Optional Setup Controls

Before resolving anything, the workflow can back up the current derived outputs and reset the MOTEL database outputs so the run starts from a clean state.


In [ ]:
setup_actions = hh.apply_setup_controls(
    harmonisation_log,
    create_motel_db_backup=create_motel_db_backup,
    reset_motel_db_outputs=reset_motel_db_outputs,
)

print("Setup actions:", setup_actions if setup_actions else ["none"])

## Step 1: Collect Unique Candidates

The harmonisation process first deduplicates the staged names so each technology, process, source, and carrier is resolved only once during the run.


In [ ]:
candidates = hh.collect_candidates(ue)

for entity_type, entity_candidates in candidates.items():
    print(f"{entity_type}: {len(entity_candidates)} unique candidates")

## Step 2: Resolve Registries

This step resolves technologies, processes, sources, and carriers against the current MOTEL registries. Exact matches are reused, semantic matches can be assisted by the LLM, and genuinely new entries are created in the registries.


In [ ]:
resolution = hh.resolve_entities_step(
    candidates,
    all_schemas,
    harmonisation_log=harmonisation_log,
)

resolved_ids = resolution["resolved_ids"]
resolved_names = resolution["resolved_names"]
resolution_status = resolution["resolution_status"]
resolution["counts_by_type"]

## Step 3: Resolve Controlled Vocabularies

Attributes and scope tokens are harmonised into controlled vocabularies so the linked entities can use stable foreign-key-style references instead of raw free text.


In [ ]:
controlled_vocab = hh.resolve_controlled_vocabulary_step(
    all_schemas,
    ue,
    full_unmapped_path=PATHS["unmapped_path"],
    rebuild_attribute_registry=True,
    harmonisation_log=harmonisation_log,
)

attr_ids = controlled_vocab["attr_ids"]
attr_names = controlled_vocab["attr_names"]
attr_status = controlled_vocab["attr_status"]
scope_ids = controlled_vocab["scope_ids"]

## Step 4: Build Linked Entities

Once all references are resolved, the workflow assembles linked entities by replacing raw names with resolved IDs for technologies, processes, carriers, sources, attributes, and scopes.


In [ ]:
linked_output = hh.build_and_save_linked_entities(
    ue,
    ue_indices,
    all_unmapped_entities,
    PATHS["unmapped_path"],
    resolved_ids,
    attr_ids,
    attr_names,
    scope_ids,
    harmonisation_log=harmonisation_log,
)

linked_entities = linked_output["linked_entities"]
today = linked_output["today"]

linked_entities[:1]

## Step 5: Save Mapping Files And Review Audit

The final stage writes the provenance and lookup maps, then prints a short audit report so the harmonisation decisions can be spot-checked.


In [ ]:
mapping_outputs = hh.save_mapping_files_step(
    ue,
    ue_indices,
    linked_entities,
    today,
    attr_ids,
    attr_names,
    attr_status,
    scope_ids,
    harmonisation_log=harmonisation_log,
)

audit_results = hh.finish_harmonisation_run(
    harmonisation_log,
    harmonisation_started,
    ue,
    linked_entities,
    attr_ids,
    ue_indices,
    audit_indices=audit_indices,
)